# Paymob Data Science Intern Assessment — Analysis & Execution Notebook
**Author:** Omar Hussain
**Date:** June 2026
**Tasks Covered:**
1. **DIGEST:** Merchant & Customer Insights (EDA, trends, timezone adjustment, MCC refund rates, spend segments)
2. **REACH:** RFM Customer Segmentation (target selection & list generation)
3. **REACH·ROI:** Campaign Response Prediction Model (leakage-free modeling, tuning, evaluation)
4. **COPILOT (Bonus):** GenAI Merchant Failure Explanations (Z-score anomaly detection, Gemini API client)



In [ ]:
# Setup imports and environment
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv

# Import our custom modules
from src.data_processor import DataProcessor
from src.segmenter import RFMSegmenter
from src.model_trainer import CampaignModelTrainer
from src.copilot_failures import detect_anomalous_merchants, get_merchant_failure_context, GeminiCopilot

# Load env variables (for Gemini API Key)
load_dotenv()

# Configure plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Setup complete and modules imported successfully.")


## 1. Data Ingestion & Cleaning (Audit Fixes)
We initialize our `DataProcessor` which applies:
* Case normalization on `payment_method`
* Dropping 534 duplicate `transaction_id` rows
* Rectifying negative values in `amount_egp` (absolute values for refunds, dropping corrupt successes)
* Labeling missing cities as "Unknown"
* Nullifying orphaned `failure_reason` records
* Timezone conversion from UTC to Africa/Cairo (UTC+2)



In [ ]:
# Instantiate Data Processor and run cleaning pipeline
processor = DataProcessor(data_dir="data")
tx_df, camp_df = processor.process_all()

print(f"Cleaned Transactions Dataset: {tx_df.shape[0]} rows, {tx_df.shape[1]} columns")
print(f"Campaign Responses Dataset: {camp_df.shape[0]} rows, {camp_df.shape[1]} columns")
tx_df.head()


## Task 1 — DIGEST: Merchant & Customer Insights
In this section, we analyze transaction trends, compute high-level KPIs, locate peak selling periods, review payment methods, study categories with high refund rates, and segment customers by total spend.



In [ ]:
# Headline KPIs
# Settled GMV = sum of successful transactions
success_tx = tx_df[tx_df['status'] == 'success']
total_gmv = success_tx['amount_egp'].sum()

# Volumes
total_volume = len(tx_df)
success_volume = len(success_tx)
failed_volume = len(tx_df[tx_df['status'] == 'failed'])
refunded_volume = len(tx_df[tx_df['status'] == 'refunded'])

# Rates
success_rate = success_volume / total_volume
failure_rate = failed_volume / total_volume
refund_rate = refunded_volume / (success_volume + refunded_volume)

# Average ticket
avg_ticket = success_tx['amount_egp'].mean()

print("--- HEADLINE KPIs ---")
print(f"Total Settled GMV: {total_gmv:,.2f} EGP")
print(f"Total Transaction Volume: {total_volume:,}")
print(f"  Success Rate: {success_rate:.2%}")
print(f"  Failure Rate: {failure_rate:.2%}")
print(f"  Refund Rate:  {refund_rate:.2%}")
print(f"Average Ticket (Success): {avg_ticket:.2f} EGP")


In [ ]:
# Monthly Trends (Cairo Local Time)
tx_df['year_month'] = tx_df['timestamp_cairo'].dt.to_period('M').astype(str)
monthly_stats = tx_df[tx_df['status'] == 'success'].groupby('year_month').agg(
    GMV=('amount_egp', 'sum'),
    Volume=('transaction_id', 'count')
).reset_index()

# Plot Monthly GMV and Volume
fig, ax1 = plt.subplots(figsize=(12, 6))

ax2 = ax1.twinx()
sns.barplot(data=monthly_stats, x='year_month', y='GMV', ax=ax1, color='royalblue', alpha=0.7, label='GMV')
sns.lineplot(data=monthly_stats, x='year_month', y='Volume', ax=ax2, color='crimson', marker='o', linewidth=2, label='Volume')

# Mark Ramadan 2024 (March-April) and Ramadan 2025 (March)
# Note: Ramadan 2024 was Mar 11 - Apr 9; Ramadan 2025 was Mar 1 - Mar 30
ax1.axvspan('2024-03', '2024-04', color='orange', alpha=0.15, label='Ramadan 24')
ax1.axvspan('2025-03', '2025-03', color='orange', alpha=0.25, label='Ramadan 25')

ax1.set_xlabel("Year-Month")
ax1.set_ylabel("Settled GMV (EGP)", color='royalblue')
ax2.set_ylabel("Transaction Count", color='crimson')
plt.title("Monthly GMV and Transaction Volume (Africa/Cairo Time)")
ax1.set_xticklabels(monthly_stats['year_month'], rotation=45)
fig.tight_layout()
plt.show()


In [ ]:
# Peak Selling Times Heatmap (Hour-of-Day x Day-of-Week)
success_tx = success_tx.copy()
success_tx['hour'] = success_tx['timestamp_cairo'].dt.hour
success_tx['day_of_week'] = success_tx['timestamp_cairo'].dt.day_name()

# Order days
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Filter online vs pos
for channel in ['online', 'pos']:
    channel_tx = success_tx[success_tx['channel'] == channel]
    pivot = channel_tx.pivot_table(index='day_of_week', columns='hour', values='transaction_id', aggfunc='count').reindex(day_order)
    
    plt.figure(figsize=(14, 5))
    sns.heatmap(pivot, cmap='YlGnBu', annot=False, fmt='d')
    plt.title(f"Peak Transaction Hours for Channel: {channel.upper()} (Africa/Cairo local time)")
    plt.ylabel("Day of Week")
    plt.xlabel("Hour of Day")
    plt.tight_layout()
    plt.show()


In [ ]:
# Payment Method Mix & Reliability
method_stats = tx_df.groupby('payment_method').agg(
    Total_Attempts=('transaction_id', 'count'),
    Success_Vol=('status', lambda x: (x == 'success').sum()),
    Failed_Vol=('status', lambda x: (x == 'failed').sum()),
    GMV=('amount_egp', lambda x: x[tx_df.loc[x.index, 'status'] == 'success'].sum())
).reset_index()

method_stats['Volume_Share'] = method_stats['Total_Attempts'] / method_stats['Total_Attempts'].sum()
method_stats['GMV_Share'] = method_stats['GMV'] / method_stats['GMV'].sum()
method_stats['Failure_Rate'] = method_stats['Failed_Vol'] / method_stats['Total_Attempts']

print("--- PAYMENT METHOD MIX ---")
print(method_stats.sort_values(by='Volume_Share', ascending=False).to_string(index=False))

# Plot share comparison
method_melted = pd.melt(method_stats, id_vars=['payment_method'], value_vars=['Volume_Share', 'GMV_Share'], var_name='Metric', value_name='Share')
plt.figure(figsize=(10, 5))
sns.barplot(data=method_melted, x='payment_method', y='Share', hue='Metric', palette='muted')
plt.title("Payment Method Share: Volume vs GMV")
plt.tight_layout()
plt.show()


In [ ]:
# Refund Rates by MCC Category
# Formula: Refund Rate = refunded / (success + refunded)
mcc_stats = tx_df[tx_df['status'].isin(['success', 'refunded'])].groupby('mcc_category').agg(
    Success_Vol=('status', lambda x: (x == 'success').sum()),
    Refunded_Vol=('status', lambda x: (x == 'refunded').sum())
).reset_index()

mcc_stats['Refund_Rate'] = mcc_stats['Refunded_Vol'] / (mcc_stats['Success_Vol'] + mcc_stats['Refunded_Vol'])
mcc_stats = mcc_stats.sort_values(by='Refund_Rate', ascending=False)

print("--- MCC REFUND RATES ---")
print(mcc_stats.to_string(index=False))

plt.figure(figsize=(12, 5))
sns.barplot(data=mcc_stats, x='mcc_category', y='Refund_Rate', palette='Reds_r')
plt.title("Refund Rate by MCC Category")
plt.ylabel("Refund Rate (Refunded / Success + Refunded)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Customer Spend Segments (Percentile Cut validation)
customer_spend = success_tx.groupby('customer_id')['amount_egp'].sum().reset_index()

# Exclude outliers when calculating percentile cuts to be statistically fair
q80 = customer_spend['amount_egp'].quantile(0.80)
q40 = customer_spend['amount_egp'].quantile(0.40)

def assign_spend_segment(spend):
    if spend >= q80:
        return 'High'
    elif spend >= q40:
        return 'Mid'
    else:
        return 'Mass'

customer_spend['Spend_Segment'] = customer_spend['amount_egp'].apply(assign_spend_segment)

# Merge back and show aggregates
segment_summary = customer_spend.groupby('Spend_Segment').agg(
    Customer_Count=('customer_id', 'count'),
    Total_GMV=('amount_egp', 'sum'),
    Avg_Spend=('amount_egp', 'mean')
).reset_index()

segment_summary['Customer_Share'] = segment_summary['Customer_Count'] / segment_summary['Customer_Count'].sum()
segment_summary['GMV_Share'] = segment_summary['Total_GMV'] / segment_summary['Total_GMV'].sum()

print("--- CUSTOMER SPEND SEGMENTS ---")
print(f"High Spend Threshold: >= {q80:,.2f} EGP")
print(f"Mid Spend Threshold:  {q40:,.2f} to {q80:,.2f} EGP")
print(f"Mass Spend Threshold: < {q40:,.2f} EGP")
print("
Segment Summary:")
print(segment_summary.to_string(index=False))


## Task 2 — REACH: Customer Segmentation (RFM)
We run our `RFMSegmenter` pipeline to compute:
* **Recency**: Days since last success relative to 2025-06-30.
* **Frequency**: Successful transaction counts.
* **Monetary**: Settled customer spend.
We assign quintiles (1-5), classify customers into business categories, and export our high-value win-back cohort (**At-Risk**) to `target_customers_at_risk.csv`.



In [ ]:
# Run RFM Segmentation
segmenter = RFMSegmenter(reference_date="2025-06-30")
rfm_raw = segmenter.calculate_rfm(tx_df)
rfm_scored = segmenter.score_rfm(rfm_raw)
rfm_segmented = segmenter.assign_segments(rfm_scored)

# Save At-Risk cohort to target_customers_at_risk.csv
at_risk_list = segmenter.generate_and_export_at_risk(rfm_segmented, output_path="target_customers_at_risk.csv")
print(f"Successfully generated target customer list with {len(at_risk_list)} at-risk targets.")

# Visualize segment sizes
plt.figure(figsize=(10, 6))
sns.countplot(data=rfm_segmented, y='Segment', order=rfm_segmented['Segment'].value_counts().index, palette='viridis')
plt.title("Distribution of Customer Segments")
plt.tight_layout()
plt.show()


## Task 3 — REACH·ROI: Campaign Response Prediction
In this task, we build a predictive model to determine whether targeted customers will respond to campaigns.
To prevent **data leakage**, all features are engineered using transactions strictly before the campaign date of `2025-04-15`.



In [ ]:
# Run ML modeling pipeline
trainer = CampaignModelTrainer(output_dir="artifacts_model")
features_df = trainer.engineer_features(tx_df, camp_df)

# Train baseline (Logistic Regression) & challenger (Random Forest) models
best_rf = trainer.train_and_evaluate(features_df)


In [ ]:
# Show the generated feature importance and evaluation curves plots inline
from PIL import Image

try:
    curves_img = Image.open("artifacts_model/model_evaluation_curves.png")
    plt.figure(figsize=(12, 6))
    plt.imshow(curves_img)
    plt.axis('off')
    plt.title("Model Performance Curves (ROC & Precision-Recall)")
    plt.show()
    
    importance_img = Image.open("artifacts_model/feature_importance.png")
    plt.figure(figsize=(10, 6))
    plt.imshow(importance_img)
    plt.axis('off')
    plt.title("Random Forest Feature Importance")
    plt.show()
except Exception as e:
    print(f"Could not load images: {e}")


## Task 4 (Bonus) — COPILOT: Explain the Failures (GenAI)
We identify merchants with statistically high failure rates (Z-score > 2, min volume 50).
Then, we pass the condensed breakdown of failure statistics to the `GeminiCopilot` to get plain-English diagnoses and support recommendations.
If `GEMINI_API_KEY` is not present in `.env`, the client runs in a structured simulator mode.



In [ ]:
# Run Anomaly Detection
anomalous_merchants = detect_anomalous_merchants(tx_df)
print("
--- ANOMALOUS MERCHANTS IDENTIFIED ---")
print(anomalous_merchants.head(5).to_string(index=False))

# Select the top anomalous merchant and get failure context
if not anomalous_merchants.empty:
    target_merchant = anomalous_merchants.iloc[0]['merchant_id']
    context = get_merchant_failure_context(target_merchant, tx_df)
    
    # Initialize and call Copilot
    copilot = GeminiCopilot()
    report = copilot.explain_merchant_failures(context)
    
    print(f"
--- COPILOT FAILURE EXPLANATION FOR MERCHANT {target_merchant} ---")
    print(report)
else:
    print("No anomalous merchants found.")
